# Units

This package provides unit conversion tools 

In [ ]:
from beamphysics.units import (
    NAMED_UNITS,
    dimension_name,
    pmd_unit,
    power_unit,
    sqrt_unit,
)
from beamphysics import particle_paths
from h5py import File

This is the basic class:

In [ ]:
help(pmd_unit)

Get a known units. These can be multiplied and divided:

In [ ]:
u1 = pmd_unit("J")
u2 = pmd_unit("m")
u1, u2, u1 / u2, u1 * u2

In [ ]:
pmd_unit("W/sqrt(Hz)")

In [ ]:
(pmd_unit("W/sqrt(m)") * pmd_unit("sqrt(m)")).simplify()

Special function for `sqrt`. It halves the dimension exponents, so the result can carry fractional dimensions:

In [ ]:
sqrt_unit(u1)

## Power notation

Units can be raised to powers using `^` notation. Integer, decimal, and fractional powers are supported:

Integer and decimal powers:

In [ ]:
pmd_unit("m^2"), pmd_unit("m^-1"), pmd_unit("m^0.5")

Fractional powers using parentheses, e.g. `m^(-3/2)`, `m^(1/2)`:

In [ ]:
pmd_unit("m^(-3/2)"), pmd_unit("m^(3/2)")

`sqrt()` notation is equivalent to `^(1/2)`:

In [ ]:
pmd_unit("sqrt(m)"), pmd_unit("m^(1/2)")

Powers work in compound expressions:

In [ ]:
pmd_unit("eV/c*m^(-1/2)")

Fractional dimensions are preserved:

In [ ]:
u = pmd_unit("m^(-3/2)")
u.unitDimension  # Length dimension is -1.5

`power_unit(u, n)` raises a unit to an arbitrary power. For a compound symbol, the exponent is distributed across each token — so `(eV*s)^2` produces `eV^2*s^2`, with the dimension multiplied through correctly (and not the malformed `eV*s^2`, which the flat left-associative parser would read as `eV*(s^2)`):


In [ ]:
power_unit(pmd_unit("eV*s"), 2), power_unit(pmd_unit("m^2*s^-1"), 3)

accepts `*` and `/`  between two known symbols.

In [ ]:
pmd_unit("W*s")

# Named units
The package knows the following units with these symbol named 

In [ ]:
NAMED_UNITS

A few common ASCII aliases are accepted as alternate spellings of the named units (the canonical symbol is preserved in the stored `unitSymbol`, so the alias and the canonical form compare equal):


In [ ]:
pmd_unit("Ohm"), pmd_unit("Ω"), pmd_unit("deg"), pmd_unit("degree")

# SI prefixes

Any known unit symbol may carry a standard SI prefix (`k`, `M`, `G`, `T`, `m`, `µ`, `n`, ...). The prefix scales `unitSI` and leaves the dimension unchanged:

In [ ]:
pmd_unit("keV"), pmd_unit("mm"), pmd_unit("GHz")

Prefixes also apply to the dimensionless angle units `rad` and `degree` (e.g. milliradians):

In [ ]:
pmd_unit("mrad"), pmd_unit("µrad")

A prefix is only accepted when the remainder is itself a known unit, and a bare prefix is **not** a unit. Symbols that are both a prefix letter and a unit keep their unit meaning — `c` is the speed of light (not centi) and `T` is tesla (not tera):

In [ ]:
pmd_unit("c"), pmd_unit("T")  # the units, not centi-/tera-

A bare prefix, or a prefixed dimensionless identity, is rejected:

In [ ]:
for s in ["k", "M", "k1"]:
    try:
        pmd_unit(s)
    except ValueError as ex:
        print(f"{s!r} -> {type(ex).__name__}: {ex}")

# `.simplify()`

The `.simplify()` method searches the internal `NAMED_UNITS` list for an equivalent unit with a simpler symbol. It can:

- collapse a product or quotient to a single named unit (`W*s` &rarr; `J`, `V*A` &rarr; `W`, `J/C` &rarr; `V`);
- reconstruct an `a*b` / `a/b` compound when no single name exists (e.g. `T*m`, `T/m`);
- attach an SI prefix when the value is an exact power-of-ten multiple of a named unit (`1000 J` &rarr; `kJ`).

Matching is float-tolerant and deterministic (the simplest symbol wins), and the resulting symbol always re-parses back to the same unit. `.simplify()` is not called automatically.

In [ ]:
u = pmd_unit("W*s")
u.simplify()

A product or quotient of named units, or a reconstructed compound:

In [ ]:
(
    (pmd_unit("V") * pmd_unit("A")).simplify(),  # -> W
    (pmd_unit("J") / pmd_unit("C")).simplify(),  # -> V
    (pmd_unit("T") * pmd_unit("m")).simplify(),  # -> T*m (no single name)
)

In [ ]:
u = pmd_unit("this should be kJ", 1e3, (2, 1, -2, 0, 0, 0, 0))
simplified = u.simplify()

# The simplified symbol re-parses back to the same unit:
simplified, pmd_unit(simplified.unitSymbol)

A purely dimensionless quantity has no meaningful named form, so `.simplify()` leaves it unchanged rather than inventing a same-dimension ratio. For example, `mrad/µrad` is just the number `1000`, and it is **not** rewritten as `kg/g`:

In [ ]:
pmd_unit("mrad/µrad").simplify()  # 1000, dimensionless -> unchanged

# Canonical symbols, equality, and validation

The parser normalizes whitespace around operators (and inside `sqrt(...)`) on the way in, so all the spellings below produce the same stored symbol — and therefore compare equal and hash identically:


In [ ]:
spellings = ["eV/c", "eV / c", "  eV/c  ", "eV /c"]
[pmd_unit(s).unitSymbol for s in spellings]

In [ ]:
a, b = pmd_unit("eV / c"), pmd_unit("eV/c")
a == b, hash(a) == hash(b)

Whitespace inside `sqrt(...)` is also normalized:


In [ ]:
pmd_unit("sqrt( m )").unitSymbol, pmd_unit("sqrt( m )") == pmd_unit("sqrt(m)")

Malformed inputs — empty operands around an operator, or repeated operators — are rejected rather than silently parsing via the identity:


In [ ]:
for s in ["m*", "/m", "m//s", "m**s", r"\sqrt{m}"]:
    try:
        pmd_unit(s)
    except ValueError as ex:
        print(f"{s!r:>14} -> ValueError: {ex}")

The openPMD standard defines `unitSI` as a non-negative conversion factor to SI, and the constructor enforces that:


In [ ]:
try:
    pmd_unit("bogus", unitSI=-1.0, unitDimension="1")
except ValueError as ex:
    print(f"ValueError: {ex}")

## Value-based equality

The openPMD standard defines a unit by its `unitSI` and `unitDimension` — the symbol is purely presentational. `pmd_unit.__eq__` follows that convention: two units compare equal when they share the same dimension and (within a small relative tolerance) the same SI factor, regardless of how their symbols were spelled.

So different syntactic spellings of the same physical unit are equal:


In [ ]:
pmd_unit("1/s") == pmd_unit("Hz"), pmd_unit("s^-1") == pmd_unit("Hz")

Arithmetic round-trips survive floating-point drift — `(eV/c) * c` reconstructs `eV` even though the intermediate float math doesn't reproduce `e_charge` bit-exactly:


In [ ]:
roundtrip = pmd_unit("eV/c") * pmd_unit("c")
roundtrip, roundtrip == pmd_unit("eV")

Same dimension but different SI scale stays unequal (`J` and `eV` are both energies, but `1 J ≠ 1 eV`):


In [ ]:
pmd_unit("J") == pmd_unit("eV"), pmd_unit("J/m") == pmd_unit("eV/m")

The hash/eq invariant is preserved — equal units share a hash bucket, so they de-duplicate correctly in sets and dicts:


In [ ]:
hash(pmd_unit("1/s")) == hash(pmd_unit("Hz")), len({pmd_unit("1/s"), pmd_unit("Hz")})

# openPMD HDF5 units

Open a file, find the particle paths from the root attributes

In [ ]:
# Pick one:
# H5FILE = 'data/bmad_particles.h5'
H5FILE = "data/distgen_particles.h5"
# H5FILE = 'data/astra_particles.h5'
h5 = File(H5FILE, "r")

ppaths = particle_paths(h5)
print(ppaths)

This points to a single particle group:

In [ ]:
ph5 = h5[ppaths[0]]
list(ph5)

Each component should have a dimension and a conversion factor to SI:

In [ ]:
d = dict(ph5["momentum/x"].attrs)
d

In [ ]:
tuple(d["unitDimension"])

This will extract the name of this dimension:

In [ ]:
dimension_name(d["unitDimension"])

# Nice arrays

In [ ]:
from beamphysics.units import nice_array

This will scale the array, and return the appropriate SI prefix:

In [ ]:
x = 1e-4
unit = "m"
nice_array(x)

In [ ]:
nice_array([-0.01, 0.01])

In [ ]:
from beamphysics.units import nice_scale_prefix

In [ ]:
nice_scale_prefix(0.009)

# TeX rendering for plot labels

`pmd_unit.to_tex()` translates the stored symbol into a matplotlib-mathtext fragment. The translation is **purely structural** — no algebraic simplification, no reordering: each atomic name is wrapped in `\mathrm{...}`, `*` becomes `{\cdot}`, `/` stays, `^N` becomes `^{N}`, and `sqrt(X)` becomes `\sqrt{...}` (recursing on the inner expression).


In [ ]:
for s in ["eV/c", "kg*m/s", "m^2", "s^-1", "m^(1/2)", "sqrt(m)", "sqrt(kg*m)"]:
    print(f"{s:>12}  ->  {pmd_unit(s).to_tex()}")

Because the translation is structural, `to_tex()` reflects the *stored* symbol — not the simplified form. A redundant compound stays compound:


In [ ]:
u = pmd_unit("m*s/m")  # dimension reduces, but the stored symbol is preserved
u.unitSymbol, u.to_tex(), u.simplify().unitSymbol

`beamphysics.labels.mathlabel` consumes this when building axis labels, so any quantity with a `pmd_unit` will render correctly as a mathtext string:


In [ ]:
from beamphysics.labels import mathlabel

mathlabel("x_bar", units=sqrt_unit(pmd_unit("m")))

# Limitations

This is a simple class for use with this package. So even simple things like the example below will fail. 

For more advanced units, use a package like Pint: https://pint.readthedocs.io/


In [ ]:
try:
    u1 / 1
except Exception as ex:
    print(f"You can't do this. It will raise {type(ex).__name__}: {ex}")